# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# NB_07 — Generación de respuestas con `ai.generate_response`

**Serie:** Microsoft Fabric AI Functions con PySpark

**Objetivo:** producir borradores de soporte por fila, comparar prompts y respuestas estructuradas, y auditar tokens y capacidad.

## 1. Requisitos y consideraciones

- Workspace en capacidad Fabric compatible y configuración de tenant habilitada.
- Sesión PySpark con AI Functions disponibles.
- Lakehouse adjunto únicamente si se desea persistir auditoría o leer archivos.
- Utilizar datos ficticios y revisar los resultados: la IA puede alterar el significado.

La función devuelve un nuevo DataFrame con el texto corregido por fila. Las entradas nulas producen resultados nulos.

## 2. Configurar el modelo y las librerías

Utilizamos el endpoint integrado de Fabric. En Runtime 2.0 (Spark 4.1), consultar la preparación temporal indicada en [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, ec96ab2e-0f40-4e17-92c8-03d6fd39da08, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


## 3. Referencia del método publicado por Microsoft

[Microsoft Learn — Use ai.generate_response with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/generate-response)

### Sintaxis documentada (adaptación de los ejemplos oficiales)

```python
df.ai.generate_response(prompt="Redacta una respuesta", output_col="response")
df.ai.generate_response(prompt="Responde a {solicitud}", is_prompt_template=True, output_col="response")
```

| Parámetro | Tipo | Uso |
|---|---|---|
| `prompt` | str | Obligatorio; instrucciones |
| `is_prompt_template` | bool | Opcional; False usa el contexto de la fila; True selecciona campos entre llaves |
| `output_col` | str | Opcional; nombre de salida |
| `error_col` | str | Opcional; nombre de errores |
| `response_format` | str/dict/BaseModel/None | Opcional; None permite formato libre |
| `col_types` | dict | Opcional; identifica columnas con rutas para multimodalidad |

La salida conserva los datos del DataFrame y agrega la respuesta. Para JSON estructurado, interpretar la cadena con `from_json`.

Formatos: `text`, `json_object` (mencionar JSON en el prompt), `json_schema` y modelos Pydantic. Los ejemplos siguientes son propios, no copias del laboratorio oficial.


## 4. Crear datos ficticios

No enviar credenciales ni datos personales. Las respuestas son borradores sujetos a revisión humana.

In [2]:
data = [(1, "Alta", "El informe de ventas no se actualiza", "En investigación"),
        (2, "Media", "Necesito acceso al tablero", "Pendiente de autorización"),
        (3, "Alta", "Falló la carga nocturna", "Equipo técnico informado"),
        (4, "Baja", "Solicito documentación del catálogo", "Disponible")]
df = spark.createDataFrame(data, ["id_ticket", "prioridad", "solicitud", "estado"])
display(df)

StatementMeta(, ec96ab2e-0f40-4e17-92c8-03d6fd39da08, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a4c8b8d1-dd34-4dc5-985d-55d78c9b6b60)

## 5. Prompt simple

Utiliza todas las columnas como contexto. Se materializa en caché para reducir reevaluaciones; una pérdida de caché puede provocar nuevas llamadas.

In [3]:
from pyspark import StorageLevel
run_id = str(uuid.uuid4())
resultado = df.ai.generate_response(
    prompt="Redacta un borrador de respuesta de soporte en español, máximo 60 palabras. Usa solo los datos disponibles; no prometas plazos ni acciones no confirmadas.",
    output_col="respuesta", error_col="error_respuesta", response_format="text"
).persist(StorageLevel.MEMORY_AND_DISK)
resultado.collect()
display(resultado)

StatementMeta(, ec96ab2e-0f40-4e17-92c8-03d6fd39da08, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09308179-6076-4372-a02f-47a5f4ba730f)

## 6. Validar resultados

Revisar errores, respuestas vacías y fidelidad a los datos.

In [4]:
resultado.printSchema()
display(resultado.select("id_ticket", "solicitud", "respuesta", "error_respuesta"))
display(resultado.groupBy(F.when(F.col("error_respuesta").isNotNull(), "ERROR").when(F.col("respuesta").isNull() | (F.length(F.trim("respuesta")) == 0), "SIN_RESPUESTA").otherwise("REVISAR").alias("validacion")).count())

StatementMeta(, ec96ab2e-0f40-4e17-92c8-03d6fd39da08, 6, Finished, Available, Finished, False)

root
 |-- id_ticket: long (nullable = true)
 |-- prioridad: string (nullable = true)
 |-- solicitud: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- respuesta: string (nullable = true)
 |-- error_respuesta: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = true)
 |    |    |    |-- minor: integer (nullable = true)
 |    |    |-- statusCode: integer (nullable = true)
 |    |    |-- reasonPhrase: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 78986e6b-e593-4c57-afe3-d71c8cbeb776)

SynapseWidget(Synapse.DataFrame, ead048ba-5b91-4090-880c-18e4ed5ffef7)

## 7. Consultar estadísticas con `ai.stats`

Las estadísticas se consultan desde el DataFrame resultante de la operación:

```python
stats_df = resultado.ai.stats
display(stats_df)
```

La documentación incluye métricas de ejecución y consumo de tokens, entre ellas `num_successful`, `num_exceptions`, `num_unevaluated`, `input_tokens`, `cached_tokens`, `output_tokens` y `reasoning_tokens`.

La disponibilidad de cada métrica depende de la respuesta del servicio. Por eso, el código consulta los campos de forma defensiva.

In [5]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI GENERATE_RESPONSE =====")
for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, ec96ab2e-0f40-4e17-92c8-03d6fd39da08, 7, Submitted, Running, Running, True)

## 8. Estimar el consumo de capacidad

Para `gpt-5-mini`, utilizaremos las tarifas publicadas para el endpoint integrado de Fabric:

| Tipo de token | CU-segundos por 1.000 tokens |
|---|---:|
| Entrada | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

La fórmula es:

$$
CU_s = \\frac{(I-C)\\times 8.40 + C\\times 0.84 + O\\times 67.23}{1000}
$$

Donde:

- \(I\): tokens de entrada.
- \(C\): tokens de entrada en caché.
- \(O\): tokens de salida.

**Alcance:** el cálculo es una estimación del consumo de las llamadas al modelo integrado. No representa el consumo total de Spark ni el consumo completo de capacidad del workspace.

Los tokens de razonamiento se muestran como dato informativo y no se agregan por separado a la fórmula.

In [ ]:
if any(stats.get(k) is None for k in ("input_tokens", "output_tokens", "cached_tokens")):
    raise RuntimeError("Faltan métricas de tokens; no se puede estimar el consumo con fiabilidad.")
input_tokens = int(stats["input_tokens"])
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

if cached_tokens > input_tokens:
    raise ValueError(
        "cached_tokens no puede ser mayor que input_tokens."
    )

INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

regular_input_tokens = input_tokens - cached_tokens

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== ESTIMACIÓN DE CONSUMO =====")
print(f"Modelo reportado: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de entrada normales: {regular_input_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento (informativo): {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.6f}")

## 9. Construir un registro de auditoría

Se conserva el esquema de los primeros notebooks. Se registra la ejecución principal sin almacenar los textos. El estado describe la ejecución técnica, no la calidad lingüística.

In [ ]:
def metric_int(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

num_input_rows = df.count()
num_successful = metric_int("num_successful")
num_exceptions = metric_int("num_exceptions")
num_unevaluated = metric_int("num_unevaluated")
num_harmful = metric_int("num_harmful")

if num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == num_input_rows:
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_07_AI_GENERATE_RESPONSE",
    "ai.generate_response",
    MODEL_NAME,
    str(stats.get("model") or MODEL_NAME),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

## 10. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 11. Ejemplos avanzados (opcionales)

Activar individualmente. Cada llamada consume capacidad adicional y se audita por separado. El prompt de plantilla limita el contexto a los campos nombrados; no usar f-string para los marcadores. El esquema JSON exige campos definidos, pero no garantiza exactitud factual.

In [ ]:
RUN_TEMPLATE = False
RUN_JSON = False
RUN_MULTIMODAL = False

def ejecutar_opcional(origen, **kwargs):
    rid = str(uuid.uuid4())
    out = origen.ai.generate_response(**kwargs).persist(StorageLevel.MEMORY_AND_DISK)
    try:
        out.collect()
        display(out)
        display(out.ai.stats)
        print("Run ID:", rid)
        # Estadísticas independientes de la auditoría principal.
        return out
    except Exception:
        out.unpersist()
        raise

if RUN_TEMPLATE:
    plantilla = ejecutar_opcional(df, prompt="Redacta un acuse de recibo para {solicitud}. Estado confirmado: {estado}. No inventes datos.", is_prompt_template=True, output_col="respuesta_plantilla", error_col="error_plantilla", response_format="text")
    plantilla.unpersist()

In [ ]:
if RUN_JSON:
    from pyspark.sql.types import BooleanType
    formato = {"type": "json_schema", "json_schema": {"name": "respuesta_soporte", "strict": True, "schema": {
        "type": "object", "properties": {"respuesta": {"type": "string"}, "requiere_revision": {"type": "boolean"}},
        "required": ["respuesta", "requiere_revision"], "additionalProperties": False}}}
    estructurado = ejecutar_opcional(df, prompt="Genera un borrador de soporte basado únicamente en los datos y marca requiere_revision como true.", response_format=formato, output_col="respuesta_json", error_col="error_json")
    esquema = StructType([StructField("respuesta", StringType()), StructField("requiere_revision", BooleanType())])
    display(estructurado.withColumn("objeto", F.from_json("respuesta_json", esquema)).select("id_ticket", "objeto.*", "error_json"))
    estructurado.unpersist()

### Multimodalidad

Utilizar `col_types` para identificar rutas. Reemplazar por una ruta accesible según la [guía multimodal](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview).

In [ ]:
if RUN_MULTIMODAL:
    archivos = spark.createDataFrame([("Files/ejemplo.pdf",)], ["file_path"])
    multimodal = ejecutar_opcional(archivos, prompt="Describe el documento sin inventar información.", col_types={"file_path": "path"}, output_col="respuesta_documento", error_col="error_documento")
    multimodal.unpersist()

## 12. Liberar caché

In [ ]:
resultado.unpersist()

## 13. Conclusiones y referencias

Se practicó generación de respuestas, plantillas y JSON estructurado. La auditoría Delta corresponde a la ejecución principal; las opcionales muestran estadísticas independientes y no se incluyen en esa tabla. Revisar las respuestas antes de utilizarlas.

- [Método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/generate-response)
- [Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Las tarifas se mantienen como referencia de la serie; verificar vigencia. El consumo estimado del modelo no incluye todo Spark. Validación local de sintaxis, no ejecución en Fabric.